# 환경설정


In [1]:
%pip install -q langchain langchain-openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 현재 환경 확인
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")


✅ 로컬 Jupyter


In [3]:
import os
import json
from pathlib import Path
from datetime import datetime
from getpass import getpass
from dotenv import load_dotenv
# langchain 패키지 가져오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 현재 작업 폴더 기준
BASE_DIR = Path.cwd()
print("현재 작업 폴더:", BASE_DIR)
print("✅ import 완료")


현재 작업 폴더: c:\workspaces\chatbot\langchain-web-chatbot
✅ import 완료


In [4]:
# OPENAI API 키 확인
load_dotenv(BASE_DIR / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ").strip()
print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


✅ API 키 OK


### 실습과제 4번


In [5]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}


In [6]:
# 페르소사 소개 사전
PERSONA_INTROS = {
    "분석가": "수치와 근거 기반 답변",
    "마케터": "비즈니스 관점과 마케팅 사례 중심 답변",
    "개발자": "코드 예시를 활용한 실용적인 답변",
    "기획자": "사용자 가치와 우선순위 중심 답변",
    "디자이너": "사용자 경험과 시각적 비유로 쉬운 답변",
}

In [7]:
def handle_command(user_input: str, current_persona: str):
    parts = user_input.strip().split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]

    if not parts:
        return current_persona, False

    cmd = parts[0]

    if cmd in ("종료", "/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    elif cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd in ("/list", "/personas"):
        print("\n사용 가능한 페르소나:")
        for name in PERSONAS:
            print(f"- {name}: {PERSONA_INTROS.get(name, '소개 미등록')}")

    elif cmd in ("/help", "/"):
        print("📖 사용법:")
        print("- 일반 질문을 입력하면 현재 페르소나가 답변합니다.")
        print("- /change <이름>: 페르소나 변경")
        print("- /persona: 현재 페르소나 확인")
        print("- /list, /personas: 이름과 한 줄 소개")
        print("- /help, /: 도움말")
        print("- 종료, /quit, /exit: 프로그램 종료")

    elif cmd == "/change":
        new_name = parts[1].strip() if len(parts) > 1 else ""

        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        print("❌ 등록된 이름을 입력하세요. 예: /change 마케터")
        print(f"사용 가능: {', '.join(PERSONAS)}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False


In [8]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"),
    ("human", "{question}"),
])

# llm은 미리 생성한 LangChain 채팅 모델
PERSONA_LLMS = {}

for name in PERSONAS:
    PERSONA_LLMS[name] = llm


## 실행


In [9]:
persona = "분석가" if "분석가" in PERSONAS else next(iter(PERSONAS))
print(f"현재 페르소나: {persona}")
print("질문을 입력하세요. (/help: 도움말)")

while True:
    user_input = input("나: ").strip()

    if not user_input:
        continue

    if user_input.startswith("/") or user_input == "종료":
        persona, should_exit = handle_command(user_input, persona)

        if should_exit:
            break
        continue

    persona_llm = PERSONA_LLMS[persona]
    chain = prompt | persona_llm | StrOutputParser()

    answer = chain.invoke({
        "role": PERSONAS[persona],
        "question": user_input,
    })

    print(answer)

현재 페르소나: 분석가
질문을 입력하세요. (/help: 도움말)
📖 사용법:
- 일반 질문을 입력하면 현재 페르소나가 답변합니다.
- /change <이름>: 페르소나 변경
- /persona: 현재 페르소나 확인
- /list, /personas: 이름과 한 줄 소개
- /help, /: 도움말
- 종료, /quit, /exit: 프로그램 종료
👋 종료합니다.
